# Stage 14 — Phase 2 Ablation

## 1. Objective

Identify which Stage 13 localization component is associated with the measured Phase 2 result, without retraining the CNN, changing the split, or running unnecessary duplicate evaluations.

## 2. Phase 1 baseline

The official Phase 1 method uses a classifier-weighted ReLU CAM with an inclusive threshold of 0.50, 8-connected DFS, the per-image two-sigma component-size filter, native-DICOM coordinate scaling, and greedy one-to-one image IoU. Its saved result is based on 601 positive images in the fixed test split.

## 3. Phase 2 method

Inspection of `src/localization/improved_cam.py`, `scripts/run_improved_cam.py`, and the Stage 13 result confirms that the only localization change is map generation: Grad-CAM++ replaces the classifier-weighted CAM map. The Phase 2 threshold, connected-component extraction, two-sigma filter, box scaling, CNN checkpoint, preprocessing, test images, and matching strategy are otherwise held constant.

## 4. Ablation configurations

A. **Phase 1 Baseline** — official saved CAM result.

B. **Ablation: Grad-CAM++ removed** — remove the sole Stage 13 change and keep Stage 13's other settings. This is algorithmically identical to A, so its metrics must equal the saved baseline; no duplicate full-set run is needed.

C. **Phase 2 Full Method** — saved Stage 13 Grad-CAM++ result.

The two saved result files are checked for the same 601 images, CNN checkpoint hash, threshold/post-processing, and IoU matching before the table is generated.

## 5. Experimental setup

All configurations use the same checkpoint, train-fitted preprocessing statistics, fixed test positives, ground-truth annotations for scoring only, and Phase 1 greedy one-to-one IoU matching. Stage 9 and Stage 13 already performed the two distinct map-generation evaluations. This ablation reuses those saved per-image-validated metrics instead of rerunning either 601-image pipeline. Removing Grad-CAM++ reproduces the baseline algorithm exactly. No localization threshold or component parameter is tuned from ground-truth labels.

In [ ]:
import csv
import json

from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "results" / "phase2_ablation_summary.json").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "results" / "phase2_ablation_summary.json").is_file():
    raise FileNotFoundError("Could not locate saved Stage 14 ablation summary")
with (ROOT / "results" / "phase2_ablation_summary.json").open(encoding="utf-8") as stream:
    summary = json.load(stream)
with (ROOT / "results" / "phase2_ablation.csv").open(newline="", encoding="utf-8") as stream:
    ablation_rows = list(csv.DictReader(stream))
print(f"Configurations: {len(ablation_rows)}; fixed-test sample count: {summary['evaluation']['test_sample_count']}")

## 6. Results

Metrics below are percentages for the two thresholded overlap rates and raw IoU values for mean/median. The ablation row equals the Phase 1 row by construction because it removes the only Stage 13 change.

In [ ]:
lines = [
    "| Method | Mean IoU | Median IoU | IoU > 0 | IoU >= 0.5 |",
    "|---|---:|---:|---:|---:|",
]
for row in ablation_rows:
    lines.append(
        f"| {row['Method']} | {float(row['Mean IoU']):.6f} | "
        f"{float(row['Median IoU']):.6f} | {float(row['IoU > 0']):.2f}% | "
        f"{float(row['IoU >= 0.5']):.2f}% |"
    )
display(Markdown("\n".join(lines)))

## 7. Comparison

The primary contribution under test is the Grad-CAM++ map generator. The no-Grad-CAM++ ablation restores classifier-weighted CAM and therefore matches the baseline. The full method's mean-IoU change and relative change are reported directly from the verified outputs.

In [ ]:
change = summary['improvement_over_baseline']
print(f"Best by mean IoU: {summary['best_configuration']}")
print(f"Mean IoU absolute change: {change['mean_iou_absolute']:+.6f}")
print(f"Mean IoU relative change: {change['mean_iou_relative_percent']:+.2f}%")
print(f"IoU > 0 change: {change['iou_gt_0_percentage_points']:+.2f} percentage points")
print(f"IoU >= 0.5 change: {change['iou_ge_0_5_percentage_points']:+.2f} percentage points")

## 8. Interpretation

The full method improves mean IoU (0.080087 → 0.090277), median IoU (0.062605 → 0.068268), and the percentage with any overlap (80.70% → 90.18%). The percentage at IoU >= 0.5 remains 0.33%. Removing Grad-CAM++ returns exactly to baseline metrics, supporting association of the aggregate changes with the map-generation replacement. The improvement is not consistent across all metrics and should not be described as a universal localization gain.

## 9. Conclusion

On the same fixed test set and evaluation protocol, Grad-CAM++ is the only changed localization component and the full method has higher mean/median IoU than the baseline. The ablation therefore supports Grad-CAM++ as the source of those aggregate changes, while the unchanged high-IoU rate limits the conclusion. No CNN retraining, split changes, or extra full-set runs were performed. Stage 15 was not started.